# Lesson 5B: Principal Component Analysis - Practical

## Introduction

In Lesson 5A, we built PCA from scratch and understood the mathematics. Now we will use **production-ready implementations** and apply PCA to fascinating real-world applications.

You are working on a face recognition system. Each face image is 64x64 pixels, which means 4,096 dimensions! Training machine learning models on such high-dimensional data is:
- Computationally expensive
- Prone to overfitting
- Difficult to visualize

But here is the insight: **Human faces lie on a much lower-dimensional manifold**. Maybe we can represent faces using just 50-100 principal components instead of 4,096 pixels.

This application is called **Eigenfaces** and was one of the first successful face recognition systems.

In this lesson, we will:
1. Use scikit-learn's PCA for production applications
2. Apply PCA to the Eigenfaces problem for face recognition
3. Learn about Kernel PCA for non-linear dimensionality reduction
4. Determine optimal number of components using explained variance
5. Use PCA for data visualization and exploration
6. Apply PCA to real datasets including handwritten digits

This lesson demonstrates production deployment of PCA.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA, KernelPCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits, fetch_olivetti_faces
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Libraries loaded successfully!")

## Scikit-Learn PCA

Scikit-learn provides an optimized PCA implementation with many useful features:

```python
from sklearn.decomposition import PCA

pca = PCA(
    n_components=2,      # Number of components to keep
    whiten=False,        # Whether to whiten data
    random_state=42      # For reproducible results
)
pca.fit(X)
```

**Key parameters**:
- `n_components`: Number of components (int) or fraction of variance (float in (0, 1))
- `whiten`: Scale components to unit variance (useful for some applications)
- `random_state`: For reproducibility when using randomized SVD

**Attributes after fitting**:
- `components_`: Principal axes in feature space
- `explained_variance_`: Variance explained by each component
- `explained_variance_ratio_`: Percentage of variance explained
- `mean_`: Per-feature empirical mean
- `n_components_`: Number of components kept

In [ ]:
# Demonstrate scikit-learn PCA on digits dataset
print("Loading handwritten digits dataset...")
digits = load_digits()
X_digits = digits.data  # 1797 samples, 64 features (8x8 images)
y_digits = digits.target

print(f"Dataset shape: {X_digits.shape}")
print(f"Number of classes: {len(np.unique(y_digits))}")
print(f"\nOriginal dimensionality: {X_digits.shape[1]}")

# Fit PCA to reduce from 64D to 2D for visualization
print("\nFitting PCA to reduce from 64D to 2D...")
pca_viz = PCA(n_components=2, random_state=42)
X_pca = pca_viz.fit_transform(X_digits)

print(f"\n✅ PCA complete!")
print(f"Transformed shape: {X_pca.shape}")
print(f"Explained variance ratio: {pca_viz.explained_variance_ratio_}")
print(f"Total variance explained: {pca_viz.explained_variance_ratio_.sum()*100:.2f}%")

# Visualize
plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_digits,
                      cmap='tab10', alpha=0.6, edgecolors='k', s=30)
plt.xlabel(f'PC1 ({pca_viz.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca_viz.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
plt.title('Handwritten Digits Projected onto First 2 Principal Components',
          fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Digit', ticks=range(10))
plt.grid(True, alpha=0.3)
plt.show()

print("\n💡 Different digits form distinct clusters in the 2D PCA space!")

## Determining Optimal Number of Components

How many components should we keep? We want to:
- Capture enough variance (typically 90-95%)
- Use as few components as possible (simplicity)

### Methods:

1. **Explained Variance Ratio**: Plot cumulative explained variance
2. **Elbow Method**: Look for the "elbow" in the scree plot
3. **Target Variance**: Keep enough components to explain X% variance (e.g., 95%)
4. **Cross-validation**: Use downstream task performance

In [ ]:
# Analyze explained variance for different numbers of components
print("Analyzing explained variance across all components...")

# Fit PCA with all components
pca_full = PCA(random_state=42)
pca_full.fit(X_digits)

# Plot explained variance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot (explained variance per component)
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
            pca_full.explained_variance_ratio_, alpha=0.7, color='steelblue')
axes[0].set_xlabel('Principal Component', fontsize=12)
axes[0].set_ylabel('Explained Variance Ratio', fontsize=12)
axes[0].set_title('Scree Plot: Variance per Component', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, 30)  # Show first 30 components

# Cumulative explained variance
cumsum = np.cumsum(pca_full.explained_variance_ratio_)
axes[1].plot(range(1, len(cumsum) + 1), cumsum, 'o-', linewidth=2, markersize=4)
axes[1].axhline(y=0.95, color='r', linestyle='--', label='95% variance')
axes[1].axhline(y=0.90, color='orange', linestyle='--', label='90% variance')
axes[1].set_xlabel('Number of Components', fontsize=12)
axes[1].set_ylabel('Cumulative Explained Variance', fontsize=12)
axes[1].set_title('Cumulative Explained Variance', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0, 40)

plt.tight_layout()
plt.show()

# Find number of components for 90% and 95% variance
n_components_90 = np.argmax(cumsum >= 0.90) + 1
n_components_95 = np.argmax(cumsum >= 0.95) + 1

print(f"\n📊 Component Analysis:")
print(f"Components for 90% variance: {n_components_90} (from 64 features)")
print(f"Components for 95% variance: {n_components_95} (from 64 features)")
print(f"Reduction: {64 - n_components_95} dimensions eliminated!")

## Eigenfaces: Face Recognition with PCA

**Eigenfaces** represent faces as linear combinations of principal component "eigenfaces".

### How it Works:

1. **Training**: Compute PCA on face images to get eigenfaces
2. **Encoding**: Project each face onto eigenface basis (low-dimensional representation)
3. **Recognition**: Compare new face's projection to known faces
4. **Reconstruction**: Reconstruct faces from eigenface coefficients

### Why it Works:

Human faces share common structure:
- Two eyes, one nose, one mouth
- Similar shapes and proportions
- Variations are mostly in details

PCA captures this shared structure in a few eigenfaces!

In [ ]:
# Load Olivetti faces dataset
print("Loading Olivetti faces dataset...")
try:
    faces = fetch_olivetti_faces(shuffle=True, random_state=42)
    X_faces = faces.data  # 400 images, 4096 features (64x64 pixels)
    y_faces = faces.target  # 40 different people

    print(f"✅ Dataset loaded!")
    print(f"Number of faces: {X_faces.shape[0]}")
    print(f"Image dimensions: {int(np.sqrt(X_faces.shape[1]))}x{int(np.sqrt(X_faces.shape[1]))}")
    print(f"Features per image: {X_faces.shape[1]}")
    print(f"Number of people: {len(np.unique(y_faces))}")

    # Display some sample faces
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    for i, ax in enumerate(axes.flat):
        ax.imshow(X_faces[i].reshape(64, 64), cmap='gray')
        ax.set_title(f'Person {y_faces[i]}')
        ax.axis('off')
    plt.suptitle('Sample Face Images', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"⚠️ Could not download faces dataset: {e}")
    print("Creating synthetic face-like data for demonstration...")
    X_faces = np.random.rand(400, 4096)
    y_faces = np.repeat(np.arange(40), 10)

In [ ]:
# Apply PCA to faces
print("Applying PCA to extract eigenfaces...")

# Fit PCA with 150 components (capturing most variance)
n_components = 150
pca_faces = PCA(n_components=n_components, whiten=True, random_state=42)
X_faces_pca = pca_faces.fit_transform(X_faces)

print(f"\n✅ Eigenfaces computed!")
print(f"Reduced from {X_faces.shape[1]} to {n_components} dimensions")
print(f"Total variance explained: {pca_faces.explained_variance_ratio_.sum()*100:.2f}%")

# Display eigenfaces (principal components)
fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    if i < len(pca_faces.components_):
        eigenface = pca_faces.components_[i].reshape(64, 64)
        ax.imshow(eigenface, cmap='gray')
        ax.set_title(f'Eigenface {i+1}\n({pca_faces.explained_variance_ratio_[i]*100:.1f}%)',
                     fontsize=10)
    ax.axis('off')
plt.suptitle('Top 18 Eigenfaces (Principal Components)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Each eigenface captures a specific facial feature variation!")
print("Face images are linear combinations of these eigenfaces.")

## Face Recognition with PCA Features

Now let us use the PCA-reduced features for face recognition. We will train a classifier using only the eigenface coefficients instead of raw pixels.

In [ ]:
# Split data for face recognition task
print("Training face recognition classifier...")

X_train, X_test, y_train, y_test = train_test_split(
    X_faces_pca, y_faces, test_size=0.25, random_state=42
)

# Train Random Forest on PCA features
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n✅ Face recognition results:")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"\nUsing only {n_components} features instead of {X_faces.shape[1]}!")
print(f"That is a {(1 - n_components/X_faces.shape[1])*100:.1f}% reduction in dimensionality")

# Show some predictions
print(f"\nSample predictions (first 10 test cases):")
for i in range(min(10, len(y_test))):
    print(f"  True: Person {y_test.iloc[i] if isinstance(y_test, pd.Series) else y_test[i]}, "
          f"Predicted: Person {y_pred[i]}, "
          f"{'✓' if y_test.iloc[i] == y_pred[i] if isinstance(y_test, pd.Series) else y_test[i] == y_pred[i] else '✗'}")

## Conclusion

### What We Learned

In this lesson, we successfully applied PCA to real-world problems:

1. **Scikit-learn PCA**: Used production-ready implementation with efficient SVD
2. **Explained variance analysis**: Determined optimal number of components
3. **Eigenfaces**: Applied PCA to face recognition problem
4. **Dimensionality reduction**: Reduced features from 4,096 to 150 while maintaining performance
5. **Data visualization**: Projected high-dimensional data to 2D for visualization

### Key Takeaways

✅ **PCA finds orthogonal directions of maximum variance**  
✅ **Use explained variance to determine how many components to keep**  
✅ **PCA is great for visualization** (project to 2D or 3D)  
✅ **PCA can improve machine learning** (fewer features, less overfitting)  
✅ **Eigenfaces show PCA captures meaningful structure** in images

### When to Use PCA

**Use PCA when:**
- You have many correlated features
- You need to visualize high-dimensional data
- You want to reduce computational cost
- You need to remove noise
- Features are continuous and roughly Gaussian

**Avoid PCA when:**
- Features are already uncorrelated
- You need to maintain interpretability (PCA components are linear combinations)
- Data has non-linear structure (use Kernel PCA or manifold learning instead)
- Categorical features dominate (PCA assumes continuous data)

### Next: Lesson 6 - Manifold Learning

PCA assumes linear structure (data lies near a hyperplane). In Lesson 6, we will learn **t-SNE** and **UMAP** which can handle non-linear manifolds like Swiss rolls and S-curves!